In [ ]:
#today i should learn about the window rolling operations and how to use it
""" windows 在使用的时候，是指定调用包括今天，或前几天在内的数，
比如说，rolling(window = 2),就是调用今天和上一个窗口的数据，如果没有上一个窗口的数据，那么返回的结果就是None。

如果写成rolling(window = '2D')，则是以每一天为间隔单位，来调用两个窗口.

'center'的作用 默认center= False 即右边对其，返回的是右边的窗格，center= True 表示居中对齐，首尾无数据

rolling(window = ‘2D’, close = 'right')这是默认的，如果不写也是默认为right,close = 'right'为右闭左开
left为左闭右开，both是两个都闭，neither 是两个都开
如果window 后面没有单位，那么默认close是both， 如果window是有单位的话，那么默认的值是‘right’

由于rolling 本身只能用std，var，和mean等简单的方法，是无法满足现实世界中的应用的，对此，我们可以使用

1.rolling.apply() 在每个滚动窗口都调用你自己定义过的函数

（1） raw = True 传numpy数组，速度快推荐，
（2） raw = False 传pandas series（慢，默认）

2.custom window rolling（自定义滚动窗口）

pandas.api.indexers （需要从这个库里边调用东西

BaseIndexer，VariableOffsetWindowIndexer，FixedForwardIndexer。

（1）BaseIndexer：可以实现任何窗口逻辑，无限制，但是有着下面的规则需要遵守。

必须实现 get_window_bounds(num_values, min_periods, center, closed) 方法
返回 (start_indices, end_indices) 两个数组，长度等于数据总行数
start_indices[i]：第 i 行窗口的起始位置
end_indices[i]：第 i 行窗口的结束位置（不包含）

（2）VariableOffsetWindowIndexer（时间偏移窗口，offset为偏移的时间，close为关闭的方式，min_periods为最少所需要的数据

（3）FixedForwardWindowIndexer（固定向前看窗口，包含当前行和未来n行：
window_size （窗口大小，包括当前行和未来n行，

还有一些我需要掌握的函数，cumsum（）累计求和

In [ ]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)
dates = pd.date_range('2024-01-01',periods = 252,freq = 'B')
tickers = ['AAPL','AMZN','MSFT','NDVA']
prices = {}

for ticker in tickers:
    returns = np.random.randn(252)*0.02
    prices[ticker] = 100*np.exp(np.cumsum(returns))
#这行代码的意思是生成连续复利的股价
df_prices = pd.DataFrame(prices,index = dates)
df_returns = df_prices.pct_change().dropna()

df_prices

#下午的任务# 【任务 1】计算 20 日滚动波动率（年化）
# 要求：使用 rolling(20).std(ddof=0) 再乘以 sqrt(252)
# 注意：std 默认 ddof=1（样本标准差），金融用 ddof=0（总体标准差）
# --- 你的代码（手写，不看答案）---

#【任务 2】计算 20 日滚动夏普比率（假设无风险利率为 0）
# 公式：年化收益率 / 年化波动率
# 要求：使用 rolling(20).mean() 和 rolling(20).std()
# --- 你的代码 ---

#【任务 3】自定义滚动函数：计算 20 日内的最大回撤（Maximum Drawdown）
# 提示：回撤就是看你能不能抗得住，最关键的指标
# 1. 回撤 = (当前价格 - 窗口内最高价) / 窗口内最高价
# 2. 最大回撤 = 回撤的最小值（最负的那个）
# 3. 必须使用 rolling(20).apply()，因为 pandas 没有内置最大回撤函数
# --- 你的代码 ---

#def max_drawdown(window):
    """
    计算窗口内的最大回撤
    window: 一个 Series，代表一个 20 天的价格窗口
    
    # 累计最高价
    rolling_max = window.cummax()
    # 回撤序列
    drawdown = (window - rolling_max) / rolling_max
    # 返回最小回撤（最负的值）
    return drawdown.min()"""

# 应用滚动窗口
# rolling_mdd = df_price.rolling(20).apply(max_drawdown)
# rolling_mdd.tail()一般默认看最后五行

# 【任务 4】对比 rolling().apply() 和内置函数的速度差异
# 使用 %%timeit 魔法命令（在 JupyterLab 中）
# 对比 rolling(20).mean() 和 rolling(20).apply(lambda x: x.mean())
# --- 你的代码 ---

# %%timeit
# df_returns.rolling(20).mean()

# %%timeit
# df_returns.rolling(20).apply(lambda x: x.mean())


In [ ]:
""" risk metrics  module
name : westli,
date: 2026-04-20
description :rolling_risk metrics for financial time series
""" 

import pandas as pd
import numpy as np
from typing import Optional, Union

class RickMetrics:
    def __init__(self,returns:pd.DataFrame):
        self.returns = returns
        self.prices = np.cumprod()*(1+returns)*100
        #(这里是用来计算离散的收益率和价格，如果是连续的话，应该写成 np.exp(np.cumsum(returns))*100

    def rolling_volatility(self,window:int = 20,annualize : bool = True) -> pd.DataFrame:

        vol = self.returns.rolling(window,min_periods = window//2).std(ddof = 0)
        #minperiods = window/2,的意思是，最少要有十个数字，才会进行计算，否则返回nan
        if annualize:
            annual_vol = vol*np.sqrt(252)
        return annual_vol
    def rolling_sharpe(self,window:int = 20,annualize : bool = True,rf_rate : float = 0.0) -> pd.DataFrame:

        rf_daily = rf_rate/252

        excess_returns = self.returns - rf_daily
        mean_excess = excess_returns.rolling(window,min_periods = window//2).mean()*252
        vol_excess = excess_returns.rolling(window,min_periods = window//2).std(ddof = 0)*np.sqrt(252)

        return mean_excess/vol_excess

    def rolling_max_drawdown(self,window:int = 20) ->pd.DataFrame:
        def calc_mdd(price_window):
            rolling_max = price_window.cummax()
            drawdown = (price_window-rolling_max)/rolling_max
            return drawdown.min()
        return self.prices.rolling(window,min_periods = window//2).apply(calc_mdd)

    def historical_var(self,confidence:float = 0.95,window:Optional[int] = None) -> pd.Series:
        if window:
            return self.returns.rolling(window).quantile(1-confidence)
        else:
            return self.returns.quantile(1-confidence)
        
        

In [1]:
# test_risk_metrics.ipynb
import sys
from pathlib import Path
sys.path.append(str(Path.home() / 'Desktop' / '代码' / 'data analysis' / '新征程'))

from processor import DataProcessor
from risk_metrics import RiskMetrics

# 加载数据
dp = DataProcessor('/Users/yuanli/Desktop/代码/data analysis/新征程/data')
dp.load_raw()
returns = dp.process_returns()

# 计算风险指标$
rm = RiskMetrics(returns)
vol = rm.rolling_volatility(20)
sharpe = rm.rolling_sharpe(20)
mdd = rm.rolling_max_drawdown(20)

print("波动率最新值:")
display(vol.tail())
print("\n夏普比率最新值:")
display(sharpe.tail())
print("\n最大回撤最新值:")
display(mdd.tail())

波动率最新值:


,AAPL,AMD,AMZN,GOOGL,JD,META,MSFT,NFLX,NVDA,OPEN,SOFI
date,,,,,,,,,,,
2025-03-25,0.302612,0.445289,0.317512,0.304546,0.516266,0.416661,0.240976,0.468706,0.658521,0.771563,0.766274
2025-03-26,0.292349,0.472225,0.323620,0.319707,0.470062,0.411664,0.244286,0.476412,0.670091,0.756521,0.760821
2025-03-27,0.294531,0.450609,0.312553,0.314124,0.479287,0.407678,0.236491,0.466730,0.609279,0.726436,0.787566
2025-03-28,0.294223,0.467550,0.335539,0.350080,0.488675,0.422461,0.254032,0.487156,0.590219,0.706023,0.781003
2025-03-31,0.303087,0.463486,0.319524,0.346607,0.469034,0.420349,0.245708,0.486811,0.512122,0.706551,0.757590



夏普比率最新值:


,AAPL,AMD,AMZN,GOOGL,JD,META,MSFT,NFLX,NVDA,OPEN,SOFI
date,,,,,,,,,,,
2025-03-25,-3.959690,3.037255,-1.184159,-0.960607,1.093713,-1.230857,-0.240670,0.785670,-0.580303,-3.123717,-0.064756
2025-03-26,-3.360900,1.590111,-2.312254,-1.581575,-0.244031,-2.749103,-1.152068,-0.280216,-2.340456,-3.744725,-1.294441
2025-03-27,-2.343363,2.164306,-1.292945,-1.312689,0.452921,-2.491649,-0.147037,0.618738,-1.244342,-3.094514,-1.873168
2025-03-28,-4.303423,1.124387,-3.452471,-3.312852,0.044609,-4.134733,-2.196397,-1.012154,-2.469235,-2.670916,-2.810397
2025-03-31,-2.715603,1.451568,-2.780237,-2.574917,0.813569,-3.585354,-1.634467,-0.868382,-0.996265,-2.674807,-2.171573



最大回撤最新值:


,AAPL,AMD,AMZN,GOOGL,JD,META,MSFT,NFLX,NVDA,OPEN,SOFI
date,,,,,,,,,,,
2025-03-25,-0.132980,-0.077430,-0.100443,-0.074801,-0.082725,-0.134811,-0.055484,-0.125378,-0.185101,-0.279221,-0.227367
2025-03-26,-0.132980,-0.049572,-0.091671,-0.074801,-0.082725,-0.127689,-0.055484,-0.125378,-0.143612,-0.223776,-0.227367
2025-03-27,-0.132980,-0.071074,-0.091671,-0.074801,-0.082725,-0.127689,-0.055484,-0.125378,-0.143612,-0.171642,-0.227367
2025-03-28,-0.122935,-0.100949,-0.075062,-0.111309,-0.082725,-0.120671,-0.055484,-0.125378,-0.098627,-0.104839,-0.176124
2025-03-31,-0.122935,-0.105130,-0.086869,-0.111309,-0.090467,-0.121250,-0.063912,-0.125378,-0.109230,-0.177419,-0.167535


In [ ]:
import sys
from pathlib import Path

# 1. 禁止生成缓存文件（永不产生__pycache__）
sys.dont_write_bytecode = True

# 2. 把当前文件夹加入搜索路径【不删除原有路径！！】
sys.path.append(str(Path.cwd()))

# 3. 导入类
from processor import DataProcessor

# 4. 初始化（你的路径）
dp = DataProcessor('/Users/yuanli/Desktop/代码/data analysis/新征程/data')

print("🎉 成功！所有问题都解决了！")